# Explore University of Mannheim contributors (OSF + Figshare)

Exploratory / driver notebook that identifies which OSF and Figshare records are linked to
**University of Mannheim** people, and assigns each a Mannheim-affiliation score.

## What it does
1. Loads the harvested metadata in `data/from_papers/osf_metadata/` and `figshare_metadata/`.
2. Flags contributors/authors that mention *Mannheim* in their profile (free-text signal).
3. Matches contributors against the Uni Mannheim employee list
   (`UniMa_Employee_List_llm_enriched_normalized_orcid_api.csv`) — **ORCID first, name match as fallback**.
4. Applies a combined record-level rule to mark each OSF/Figshare item as "from Uni Mannheim".
5. Runs the shared scorer (`from scoring import run_all`) to produce the scored tables
   `osf_unimannheim_scored_potential.csv` and `figshare_unimannheim_scored_potential.csv`,
   which `code/from_papers/build_unified_metadata.py` later merges into the unified librarian file.

## Inputs
- `data/from_papers/osf_metadata/*.json`, `figshare_metadata/*.json` (harvested metadata)
- `data/from_papers/UniMa_Employee_List_llm_enriched_normalized_orcid_api.csv` (ORCID -> name)
- `code/from_papers/scoring.py` (`run_all`)

## ⚠️ Personal data
This notebook reads and produces **personal data** (names, emails, ORCIDs, and automated
affiliation verdicts about named individuals). The employee list and the scored outputs are
**git-ignored** and must not be committed. Clear cell outputs before sharing this notebook so
no names end up embedded in saved output.


In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

# Notebooks often run with cwd = notebooks/, so try both locations.
osf_metadata_dir_candidates = [
    Path("data/from_papers/osf_metadata"),
    Path("../data/from_papers/osf_metadata"),
]
osf_metadata_dir = next((p for p in osf_metadata_dir_candidates if p.exists()), osf_metadata_dir_candidates[0])

paths = sorted(osf_metadata_dir.glob("osf_*.json"))
len(paths)


In [16]:
MANNHEIM_RE = re.compile(r"\bmannheim\b", re.IGNORECASE)

def iter_text_fields(x):
    """Yield all string values nested inside dict/list structures."""
    if x is None:
        return
    if isinstance(x, str):
        yield x
        return
    if isinstance(x, dict):
        for v in x.values():
            yield from iter_text_fields(v)
        return
    if isinstance(x, list):
        for v in x:
            yield from iter_text_fields(v)
        return

def contributor_mentions_mannheim(person: dict):
    """Check employment/education text for Mannheim; return (flag, matched_snippets)."""
    snippets = []
    for field in ("employment", "education"):
        for s in iter_text_fields(person.get(field)):
            if MANNHEIM_RE.search(s):
                snippets.append(s)
    return (len(snippets) > 0), sorted(set(snippets))

rows = []
for p in paths:
    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
    except UnicodeDecodeError:
        obj = json.loads(p.read_text(encoding="cp1252"))

    osf_id = obj.get("osf_id")
    title = ((obj.get("resource") or {}).get("title") if isinstance(obj.get("resource"), dict) else "")

    identifiers = obj.get("identifiers") or []
    doi_val = ""
    if isinstance(identifiers, list):
        for it in identifiers:
            if isinstance(it, dict) and str(it.get("category") or "").lower() == "doi" and it.get("value"):
                doi_val = str(it.get("value"))
                break

    people = (((obj.get("contributors") or {}).get("people")) if isinstance(obj.get("contributors"), dict) else [])
    if not isinstance(people, list):
        people = []

    for person in people:
        if not isinstance(person, dict):
            continue
        hit, snippets = contributor_mentions_mannheim(person)
        rows.append(
            {
                "osf_id": osf_id,
                "title": title,
                "doi": doi_val,
                "user_id": person.get("user_id"),
                "full_name": person.get("full_name"),
                "given_name": person.get("given_name"),
                "family_name": person.get("family_name"),
                "mannheim": hit,
                "mannheim_snippets": " | ".join(snippets[:10]),
            }
        )

df = pd.DataFrame(rows)
df.head()


,osf_id,title,doi,user_id,full_name,given_name,family_name,mannheim,mannheim_snippets
0,24xyq,Why do (some) citizens find pleasure in politics?,,vur7x,Alexander Wuttke,Alexander,Wuttke,True,Mannheim Centre for European Social Research |...
1,25mzu,Measuring Binding Effects in Event-Based Episo...,,2dr3b,Marcel R. Schreiner,Marcel,Schreiner,True,University of Mannheim
2,25mzu,Measuring Binding Effects in Event-Based Episo...,,p5n3b,Thorsten Meiser,Thorsten,Meiser,False,
3,2djpa,Can Physical Attractiveness Close the Immigran...,,j2vr9,Joshua Hellyer,Joshua,Hellyer,True,Mannheim Centre for European Social Research
4,2dtps,Too Good To Be Liked - When and How Prosocial ...,,m2fqx,Lucia Lou-Anne Boileau,Lucia,Boileau,False,


In [17]:
# Summary: contributors mentioning Mannheim
total_contrib = len(df)
mannheim_contrib = int(df["mannheim"].fillna(False).sum())
unique_people = df[["user_id"]].dropna().nunique().iloc[0] if "user_id" in df.columns else None
unique_mannheim_people = df[df["mannheim"]][["user_id"]].dropna().nunique().iloc[0] if "user_id" in df.columns else None

summary = pd.DataFrame(
    {
        "count": [total_contrib, mannheim_contrib, unique_people, unique_mannheim_people],
    },
    index=[
        "contributors_total_rows",
        "contributors_mentions_mannheim",
        "unique_user_ids",
        "unique_user_ids_mentions_mannheim",
    ],
)

summary


,count
contributors_total_rows,772
contributors_mentions_mannheim,144
unique_user_ids,410
unique_user_ids_mentions_mannheim,48


In [18]:
# Project-level view: which OSF records have at least 1 contributor mentioning Mannheim?
by_project = (
    df.groupby(["osf_id", "title", "doi"], dropna=False)["mannheim"]
    .any()
    .reset_index(name="any_contributor_mannheim")
)

by_project["any_contributor_mannheim"].value_counts(dropna=False)


any_contributor_mannheim
False    173
True     113
Name: count, dtype: int64

In [19]:
# Inspect matches
matches = df[df["mannheim"]].sort_values(["osf_id", "full_name"], na_position="last")
matches[["osf_id", "title", "doi", "full_name", "user_id", "mannheim_snippets"]].head(50)


,osf_id,title,doi,full_name,user_id,mannheim_snippets
0,24xyq,Why do (some) citizens find pleasure in politics?,,Alexander Wuttke,vur7x,Mannheim Centre for European Social Research |...
1,25mzu,Measuring Binding Effects in Event-Based Episo...,,Marcel R. Schreiner,2dr3b,University of Mannheim
3,2djpa,Can Physical Attractiveness Close the Immigran...,,Joshua Hellyer,j2vr9,Mannheim Centre for European Social Research
19,38zp2,Ideological self-selection in online news expo...,,Sebastian Stier,5szxn,University of Mannheim
22,3b59h,"Sievert, Vogel, &amp; Feeney (2022): Formaliza...",,Martin Sievert,r4ntf,University of Mannheim
26,3fujm,Predicting the memorability of scene pictures ...,,Arndt Bröder,5rygf,University of Mannheim
24,3fujm,Predicting the memorability of scene pictures ...,,Sofia Navarro-Baez,bfyz8,University of Mannheim
29,3pkmf,Agency Effects on the Binding of Event Element...,,Arndt Bröder,5rygf,University of Mannheim
28,3pkmf,Agency Effects on the Binding of Event Element...,,Marcel R. Schreiner,2dr3b,University of Mannheim
31,3qbvx,Symbolic Gender Representation of Coproduction...,,Martin Sievert,r4ntf,University of Mannheim


In [20]:
# Fallback check using Uni Mannheim employee list (ORCID preferred, name match as backup)

employee_candidates = [
    Path("data/from_papers/UniMa_Employee_List_llm_enriched_normalized_orcid_api.csv"),
    Path("../data/from_papers/UniMa_Employee_List_llm_enriched_normalized_orcid_api.csv"),
]
employee_path = next((p for p in employee_candidates if p.exists()), employee_candidates[0])

# Excel can save this as cp1252
try:
    emp = pd.read_csv(employee_path, dtype=str)
except UnicodeDecodeError:
    emp = pd.read_csv(employee_path, dtype=str, encoding="cp1252")

emp.shape

(1324, 33)

In [21]:
import unicodedata

ORCID_RE = re.compile(r"\b\d{4}-\d{4}-\d{4}-\d{3}[0-9X]\b")

def norm_orcid(x: str) -> str:
    if x is None:
        return ""
    s = str(x).strip()
    if not s:
        return ""
    m = ORCID_RE.search(s)
    return (m.group(0) if m else "").upper()

def norm_name(x: str) -> str:
    s = "" if x is None else str(x)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower().strip()
    s = re.sub(r"[^a-z\s\-']+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s

# Build Uni Mannheim ORCID set (union of likely ORCID columns)
orcid_cols = [c for c in ["orcid_api", "orcid_m", "orcid_id_llm"] if c in emp.columns]
emp_orcids = set()
for c in orcid_cols:
    emp_orcids.update({norm_orcid(v) for v in emp[c].dropna().tolist()})
emp_orcids.discard("")

# Build Uni Mannheim name set (first+last if available; else parse full_name)
first_col = "first_name_llm" if "first_name_llm" in emp.columns else None
last_col = "last_name_llm" if "last_name_llm" in emp.columns else None

name_pairs = set()
if first_col and last_col:
    for f, l in zip(emp[first_col].fillna(""), emp[last_col].fillna("")):
        f2, l2 = norm_name(f), norm_name(l)
        if f2 and l2:
            name_pairs.add((f2, l2))

# also add parsed full_name_m as fallback
if "full_name_m" in emp.columns:
    for full in emp["full_name_m"].fillna(""):
        parts = norm_name(full).split(" ")
        if len(parts) >= 2:
            name_pairs.add((parts[0], parts[-1]))

len(emp_orcids), len(name_pairs)

(423, 1393)

In [22]:
# Extract ORCID (if present) from OSF contributor profiles in the merged JSONs

def get_orcid_from_person(person: dict) -> str:
    if not isinstance(person, dict):
        return ""
    ext = person.get("external_identity")
    if isinstance(ext, dict):
        # common OSF shape: {"ORCID": {"id": "0000-...", "status": "VERIFIED"}}
        orcid_block = ext.get("ORCID")
        if isinstance(orcid_block, dict):
            return norm_orcid(orcid_block.get("id"))
        # sometimes already a string
        if "ORCID" in ext and isinstance(ext.get("ORCID"), str):
            return norm_orcid(ext.get("ORCID"))
    return ""

rows2 = []
for p in paths:
    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
    except UnicodeDecodeError:
        obj = json.loads(p.read_text(encoding="cp1252"))

    osf_id = obj.get("osf_id")
    title = ((obj.get("resource") or {}).get("title") if isinstance(obj.get("resource"), dict) else "")

    people = (((obj.get("contributors") or {}).get("people")) if isinstance(obj.get("contributors"), dict) else [])
    if not isinstance(people, list):
        people = []

    for person in people:
        if not isinstance(person, dict):
            continue
        orcid = get_orcid_from_person(person)
        rows2.append(
            {
                "osf_id": osf_id,
                "title": title,
                "user_id": person.get("user_id"),
                "given_name": person.get("given_name"),
                "family_name": person.get("family_name"),
                "full_name": person.get("full_name"),
                "orcid": orcid,
            }
        )

df_people = pd.DataFrame(rows2)

df_people.head()

,osf_id,title,user_id,given_name,family_name,full_name,orcid
0,24xyq,Why do (some) citizens find pleasure in politics?,vur7x,Alexander,Wuttke,Alexander Wuttke,0000-0002-9579-5357
1,25mzu,Measuring Binding Effects in Event-Based Episo...,2dr3b,Marcel,Schreiner,Marcel R. Schreiner,
2,25mzu,Measuring Binding Effects in Event-Based Episo...,p5n3b,Thorsten,Meiser,Thorsten Meiser,
3,2djpa,Can Physical Attractiveness Close the Immigran...,j2vr9,Joshua,Hellyer,Joshua Hellyer,0000-0001-9034-5640
4,2dtps,Too Good To Be Liked - When and How Prosocial ...,m2fqx,Lucia,Boileau,Lucia Lou-Anne Boileau,0000-0003-4085-0918


In [23]:
# Match contributors to Uni Mannheim list:
# 1) ORCID exact match (strong)
# 2) Name (given+family) match (weaker)

def name_key(row) -> tuple[str, str]:
    g = norm_name(row.get("given_name") or "")
    f = norm_name(row.get("family_name") or "")
    if g and f:
        return (g, f)
    # fallback: parse full_name as first+last
    parts = norm_name(row.get("full_name") or "").split(" ")
    return (parts[0], parts[-1]) if len(parts) >= 2 else ("", "")

people = df_people.copy()
people["orcid_match"] = people["orcid"].apply(lambda o: bool(o) and o in emp_orcids)
people["name_pair"] = people.apply(name_key, axis=1)
people["name_match"] = people["name_pair"].apply(lambda k: k in name_pairs if k != ("", "") else False)
people["unimannheim_employee"] = people["orcid_match"] | people["name_match"]

people[["orcid_match", "name_match", "unimannheim_employee"]].sum()

orcid_match             114
name_match              245
unimannheim_employee    245
dtype: int64

In [24]:
# Project-level: mark OSF records as "from Uni Mannheim" if any contributor matches the employee list
by_project_emp = (
    people.groupby(["osf_id", "title"], dropna=False)["unimannheim_employee"]
    .any()
    .reset_index(name="any_contributor_in_employee_list")
)

by_project_emp["any_contributor_in_employee_list"].value_counts(dropna=False)

any_contributor_in_employee_list
True     176
False    110
Name: count, dtype: int64

In [25]:
# Combined rule at OSF-record level:
# True if ANY contributor either (a) mentions Mannheim in OSF employment/education OR (b) matches employee list.

# (a) from earlier Mannheim snippet scan
by_project_osf_aff = (
    df.groupby(["osf_id", "title", "doi"], dropna=False)["mannheim"]
    .any()
    .reset_index(name="any_contributor_mentions_mannheim_in_osf_profile")
)

# (b) from employee list match
by_project_emp2 = by_project_emp.copy()

combined = by_project_osf_aff.merge(by_project_emp2, on=["osf_id", "title"], how="outer")
combined["any_contributor_mentions_mannheim_in_osf_profile"] = combined[
    "any_contributor_mentions_mannheim_in_osf_profile"
].fillna(False)
combined["any_contributor_in_employee_list"] = combined["any_contributor_in_employee_list"].fillna(False)

combined["any_unimannheim"] = combined["any_contributor_mentions_mannheim_in_osf_profile"] | combined[
    "any_contributor_in_employee_list"
]

combined[["any_unimannheim", "any_contributor_mentions_mannheim_in_osf_profile", "any_contributor_in_employee_list"]].sum()

any_unimannheim                                     207
any_contributor_mentions_mannheim_in_osf_profile    113
any_contributor_in_employee_list                    176
dtype: int64

In [26]:
# How many OSF records are Uni Mannheim by the combined rule?
combined["any_unimannheim"].value_counts(dropna=False)

any_unimannheim
True     207
False     79
Name: count, dtype: int64

In [27]:
# List the OSF records marked as Uni Mannheim by combined rule
combined_true = combined[combined["any_unimannheim"]].sort_values(["osf_id"], na_position="last")
combined_true[["osf_id", "title", "doi", "any_contributor_mentions_mannheim_in_osf_profile", "any_contributor_in_employee_list"]].head(100)

,osf_id,title,doi,any_contributor_mentions_mannheim_in_osf_profile,any_contributor_in_employee_list
0,24xyq,Why do (some) citizens find pleasure in politics?,,True,False
1,25mzu,Measuring Binding Effects in Event-Based Episo...,,True,True
2,2djpa,Can Physical Attractiveness Close the Immigran...,,True,False
3,2dtps,Too Good To Be Liked - When and How Prosocial ...,,False,True
4,2jgy3,Call me maybe: Risk factors of impaired social...,,False,True
...,...,...,...,...,...
126,g7aqh,"Replication files for ""Distinct boundaries? Pr...",,True,True
128,gmkaj,Nailing down the perceptual explanation of the...,,True,True
129,gpmej,The motherhood penalty in financial resources ...,,False,True
130,gqb4y,"Different Styles, Different Times",,True,False


In [ ]:
# Score OSF + Figshare for Uni Mannheim in one step.
# All logic lives in the single shared module code/from_papers/scoring.py — both sources use
# the SAME scorer (employee-list match by ORCID->name + tenure check). Edit rules there.
import sys
from pathlib import Path

for _cand in [Path("code/from_papers"), Path("../code/from_papers")]:
    if _cand.exists():
        sys.path.insert(0, str(_cand.resolve()))
        break

import importlib
import scoring
importlib.reload(scoring)  # pick up edits to scoring.py without restarting the kernel

# Writes:
#   data/from_papers/osf_unimannheim_scored_potential.csv
#   data/from_papers/figshare_unimannheim_scored_potential.csv
osf_scored, figshare_scored = scoring.run_all()

print("\nOSF verdicts:")
print(osf_scored["verdict"].value_counts().to_string())
print("\nFigshare verdicts:")
print(figshare_scored["verdict"].value_counts().to_string())
osf_scored.head(20)


In [29]:
# Inspect matched contributors (ORCID matches first)
# Uses `people` from the employee-matching cell (cell 8). Re-applies matching if you skipped that cell.
inspect_people = people.copy() if "people" in globals() and isinstance(people, pd.DataFrame) else df_people.copy()

if "unimannheim_employee" not in inspect_people.columns:
    inspect_people["orcid_match"] = inspect_people["orcid"].apply(lambda o: bool(o) and o in emp_orcids)
    inspect_people["name_pair"] = inspect_people.apply(name_key, axis=1)
    inspect_people["name_match"] = inspect_people["name_pair"].apply(
        lambda k: k in name_pairs if k != ("", "") else False
    )
    inspect_people["unimannheim_employee"] = inspect_people["orcid_match"] | inspect_people["name_match"]

matched = inspect_people[inspect_people["unimannheim_employee"]].copy()
matched = matched.sort_values(["orcid_match", "osf_id", "family_name", "given_name"], ascending=[False, True, True, True])
matched[["osf_id", "title", "full_name", "given_name", "family_name", "orcid", "orcid_match", "name_match"]].head(50)


,osf_id,title,full_name,given_name,family_name,orcid,orcid_match,name_match
5,2jgy3,Call me maybe: Risk factors of impaired social...,Stefan Janke,Stefan,Janke,0000-0003-1799-8850,True,True
7,2pr76,Replicating the connection between hindsight b...,Barbara Kreis,Barbara,Kreis,0000-0002-8520-6475,True,True
12,327s4,Development of a short scale for assessing aca...,Stefan Janke,Stefan,Janke,0000-0003-1799-8850,True,True
17,34dvr,Real-World Estimation Taps Into Basic Numeric ...,Barbara Kreis,Barbara,Kreis,0000-0002-8520-6475,True,True
19,38zp2,Ideological self-selection in online news expo...,Sebastian Stier,Sebastian,Stier,0000-0002-1217-5778,True,True
27,3gvh7,Support for a legal right to work from home,Katja Möhring,Katja,Möhring,0000-0002-3742-5374,True,True
41,3w7ra,Effects of Performance Goals and Social Norms...,Stefan Janke,Stefan,Janke,0000-0003-1799-8850,True,True
44,45vud,Mainstreaming the populist radical right?,Sebastian Stier,Sebastian,Stier,0000-0002-1217-5778,True,True
47,45xz3,Assessing dissimilarity of employment history ...,Katja Möhring,Katja,Möhring,0000-0002-3742-5374,True,True
51,4ym7p,Diffusion of Tax-Related Communication on Soci...,Žiga Puklavec,Žiga,Puklavec,0000-0001-5158-8842,True,True
